# 06 Model E-invariant (adversarial) and the leakage probe

CLAUDE.md Decision Log, 2026-11-19: E-invariant = A's MLP split at its 512-wide hidden layer (z); a race adversary (MLP, lr 1e-2, 5 steps per encoder step) is trained on z in alternation with the encoder + age head, whose loss is L1(age) + lambda x (-CE(race | z) or KL(base rates || adversary)). Grid: lr {1e-3, 3e-4} x wd {1e-4, 1e-2} x lambda {0.1, 0.3, 1, 3} x target {max_ce, prior} = 32 settings per fold; tuning rule v3 (final). After every refit a fresh logistic-regression probe predicts race from z (balanced accuracy, chance 0.20), also on A's z and raw CLIP. Holm family = E_inv, F1, F2 (the proposal's conditions); B, C, C2 reported as context with unadjusted p. The family-of-9 comparisons file is archived first. GPU: T4 strongly recommended (estimate 1-2 h; CPU would take many hours). Charts only; no images are opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, just slower)')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Preconditions and archive of the family-of-9 comparisons

In [ ]:
A = json.loads((DRIVE_ROOT / 'results/models/A.json').read_text())
assert A['git']['dirty'] is False
assert sha256(DRIVE_ROOT / A['oof_file']['drive_path']) == A['oof_file']['sha256'], "A's OOF file changed"
assert done('utkface__clip_vitl14'), 'UTKFace embeddings missing or changed on Drive'
import shutil
R = DRIVE_ROOT / 'results'
arch = R / 'comparisons_family9.json'
if not arch.exists():
    old = json.loads((R / 'comparisons.json').read_text())
    assert len(old['family']) == 9, old['family']
    shutil.copy(R / 'comparisons.json', arch)
    print('archived comparisons.json (family of 9) ->', arch.name)
else:
    print('family-of-9 comparisons already archived')

## 2. Model E-invariant (32 settings x 5 folds, then refit + leakage probe)

In [ ]:
!python scripts/train_model.py --model E_inv
d = save_manifest('models/E_inv.json')
for f in d['folds']:
    lp = f['leakage_probe']
    print(f['test_fold'], f['chosen'], f['epochs'], 'ep | WG', round(f['inner_val_worst_group_tpr'], 3), 'allowed', f['inner_val_feasible'],
          '| test MAE', round(f['test_mae'], 2), '| ref A MAE diff', round(f['reference_A']['mae_diff_vs_A_record'], 4),
          '| A refit test MAE diff', round(lp.get('A_refit_test_mae_diff_vs_A_record', float('nan')), 4))
    print('   probe bal acc  E', round(lp['bal_acc_model'], 3), ' A', round(lp['bal_acc_A'], 3), ' CLIP', round(lp['bal_acc_clip_raw'], 3),
          '| allowed settings', f['n_trials_feasible_at_best'], '/ 32')
lp = d['leakage_probe']
print('\npooled leakage probe (balanced accuracy, chance', lp['chance'], ')')
for n, r in lp['bal_acc'].items():
    print(f"  {n:9s} {r['est']:.3f} [{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]")
for n, r in lp['diff'].items():
    print(f"  {n:20s} {r['est']:+.3f} [{r['ci'][0]:+.3f}, {r['ci'][1]:+.3f}]")

## 3. Training stability: adversary balanced accuracy per epoch (refits)

A collapse to ~0.20 with a probe still high means the adversary was fooled, not that race was removed.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for f in d['folds']:
    ri = f['refit_info']
    lab = f"fold {f['test_fold']} (lambda {f['chosen']['adv_lambda']}, {f['chosen']['adv_target']})"
    ax[0].plot(range(1, len(ri['adv_bal_acc_by_epoch']) + 1), ri['adv_bal_acc_by_epoch'], label=lab)
    ax[1].plot(range(1, len(ri['adv_ce_by_epoch']) + 1), ri['adv_ce_by_epoch'])
ax[0].axhline(0.2, ls='--', c='grey', lw=1); ax[0].set(xlabel='epoch', ylabel='adversary balanced accuracy', ylim=(0, 1))
ax[1].set(xlabel='epoch', ylabel='adversary cross-entropy'); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 4. Holm-adjusted verdicts vs A (family E_inv, F1, F2; interim until F1, F2 run; B, C, C2 as context)

In [ ]:
!python scripts/compare_models.py
save_manifest('comparisons.json')

## 5. Aggregate outputs for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step10_E_outputs.zip'
files = [R / 'models/E_inv.json', R / 'comparisons.json', R / 'comparisons_family9.json']
with zipfile.ZipFile(out, 'w') as z:
    for f in files:
        z.write(f, f.relative_to(DRIVE_ROOT))
print(out, out.stat().st_size, 'bytes, sha256', sha256(out))